# 02 — Encode SigLIP toàn bộ keyframe (resume theo shard)

**Phụ trách:** Tuấn Anh (N1) · **Mốc:** M3 · **GPU:** CẦN (Runtime → Change runtime type → T4)

Chạy được trên NHIỀU tài khoản Colab song song: mỗi người nhận một dải shard
(đặt `SHARD_START`/`SHARD_END` khác nhau). Ngắt phiên giữa chừng không sao —
chạy lại sẽ bỏ qua shard đã xong.

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
%pip -q install transformers torch pillow
import torch; assert torch.cuda.is_available(), 'Chưa bật GPU!'

# === CONFIG ===
DRIVE_ROOT   = '/content/drive/MyDrive/AIC2026'
KEYFRAME_DIR = f'{DRIVE_ROOT}/data/keyframes'      # cấu trúc: keyframes/<video_id>/<frame>.jpg
OUT_DIR      = f'{DRIVE_ROOT}/outputs/siglip_shards'
MODEL_ID     = 'google/siglip-so400m-patch14-384'
SHARD_SIZE   = 2000      # ảnh / shard (~2-3 phút/shard trên T4)
SHARD_START, SHARD_END = 0, 9999   # ← MỖI TÀI KHOẢN SỬA DẢI NÀY (chia ca theo bảng của Dũng)
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# === Danh sách ảnh (thứ tự cố định cho mọi tài khoản) ===
import glob, os
images = sorted(glob.glob(f'{KEYFRAME_DIR}/*/*.jpg'))
print(f'{len(images)} keyframe, {(len(images)-1)//SHARD_SIZE + 1} shard')

def kf_id(path):
    video_id = os.path.basename(os.path.dirname(path))
    frame = os.path.splitext(os.path.basename(path))[0]
    return f'{video_id}#{frame}'

In [ ]:
# === Encode theo shard, bỏ qua shard đã có trên Drive ===
from transformers import AutoProcessor, AutoModel
from PIL import Image
import numpy as np, torch

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float16).cuda().eval()
BATCH = 64

n_shards = (len(images) - 1) // SHARD_SIZE + 1
for s in range(max(0, SHARD_START), min(n_shards, SHARD_END + 1)):
    out_npy = f'{OUT_DIR}/shard_{s:05d}.npy'
    if os.path.exists(out_npy):
        continue   # đã xong từ phiên trước / tài khoản khác
    chunk = images[s*SHARD_SIZE : (s+1)*SHARD_SIZE]
    vecs = []
    for b in range(0, len(chunk), BATCH):
        imgs = [Image.open(p).convert('RGB') for p in chunk[b:b+BATCH]]
        inputs = processor(images=imgs, return_tensors='pt').to('cuda')
        with torch.no_grad():
            v = model.get_image_features(**inputs)
        vecs.append(v.float().cpu().numpy())
    np.save(out_npy, np.concatenate(vecs).astype('float16'))
    with open(f'{OUT_DIR}/shard_{s:05d}.ids', 'w') as f:
        f.write('\n'.join(kf_id(p) for p in chunk))
    print(f'✓ shard {s}/{n_shards}')
print('XONG dải được giao.')

In [ ]:
# === (Chạy 1 lần khi ĐỦ shard) gộp thành FAISS index ===
%pip -q install faiss-cpu
import faiss, numpy as np, glob
shards = sorted(glob.glob(f'{OUT_DIR}/shard_*.npy'))
print(len(shards), 'shard')
X = np.concatenate([np.load(p).astype('float32') for p in shards])
ids = []
for p in shards:
    ids += open(p.replace('.npy', '.ids')).read().splitlines()
faiss.normalize_L2(X)
index = faiss.IndexFlatIP(X.shape[1]); index.add(X)
out = f'{DRIVE_ROOT}/outputs/index_v2_siglip'
import os; os.makedirs(out, exist_ok=True)
faiss.write_index(index, f'{out}/faiss_siglip.index')
np.save(f'{out}/keyframe_ids.npy', np.array(ids))
print('Đã lưu', out, '—', index.ntotal, 'vector')